# **Hands-on Session 1: Downloading & Simple Analysis of Great Lakes Operational Forecast System (GLOFS)**

In this hands-on session, we will walk through the process of downloading, reading, and visualizing the unstructured mesh hydrodynamic model outputs from the Lake Erie Operational Forecast System (LEOFS), which is part of NOAA's Great Lakes Operational Forecast System (GLOFS). GLOFS is based on [the Finite Volume Community Ocean Model (FVCOM)](https://etchellsfleet27.com/wp-content/uploads/2020/06/FVCOM_User_Manual_v3.1.6.pdf), a hydrodynamic model used to simulate water circulation, temperature, and salinity in oceans and other large water bodies, including the Great Lakes. Like other hydrodynamic and ocean models, FVCOM is based on governing equations known as the primitive equations. These are a set of approximated equations used to describe the motion of water in shallow-water systems, where “shallow” means that the vertical length scale is much smaller than the horizontal length scale. This assumption applies to most oceanic and lake circulation systems.



* **Estimated Time:** 1 hour for the core part. 1.5 hour including the optional analysis.
* **Prerequisites:** Basic Python plus some familiarity with NumPy/Matplotlib. No prior FVCOM/GLOFS experience required.
* **What you'll need:** Python Environment (e.g., Google Colab, Jupyter Notebook)

| Part | Topic | Approx. time |
|------|-------|--------------|
| **Part 1** | Setup, download, and open a GLOFS file | 10 min |
| **Part 2** | Understand the FVCOM unstructured mesh | 10 min |
| **Part 3** | Plot mesh and surface temperature | 15 min |
| **Part 4** | Overlay surface velocity vectors | 15 min |
| **Optional** | Vertical transect of temperature | 10–20 min |

# **Part 1	Setup, download, and open a GLOFS file**
## **1.1 Importing Libraries**

Before we do anything else, we import the core libraries used throughout this notebook. If you are using Jupyter Notebook or another Python environment, you might have to install boto3 and botocore in advance. If you are using Google Colab, we'll mount your Google Drive so that downloaded files can be stored to your Google Drive folder. Otherwise, downloaded files will be stored in a temporary area and will be removed when you end the Google Colab session.

In [ ]:
# Install required packages.
# boto3 is used to access NOAA files on AWS S3.
! pip install boto3
! pip install botocore
import numpy as np
import pandas as pd
import xarray as xr
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import os
from pathlib import Path

use_google_drive = True

if use_google_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    save_dir = Path("/content/drive/MyDrive/GL_env_data/handson1")
else:
    save_dir = Path("/content/GL_env_data/handson1")


# Create main folder.
save_dir.mkdir(parents=True, exist_ok=True)

print(f"Files will be saved in: {save_dir}")



## **1.2 Downloading Data**
Download Great Lakes Operational Forecast System (GLOFS) output files from Amazon s3 bucket. You can specify year, month, and day as you like. If you'd like, you can also specify a system for another Great Lake. The file format of GLOFS outputs is [NetCDF (Network Common Data Form)](https://www.unidata.ucar.edu/software/netcdf).


| Abbreviation| Full System Name |
|-------------|--------------|
| leofs | Lake Erie Operational Forecast System |
| lmhofs | Lake Michigan-Huron Operational Forecast System |
| lsofs | Lake Superior Operational Forecast System |
| loofs | Lake Ontario Operational Forecast System |


In [ ]:

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))


year=2026
month=5
day=1
systemname='leofs'

yearstr=str(year).zfill(4)
monstr=str(month).zfill(2)
daystr=str(day).zfill(2)


# Create folder if it does not exist
os.makedirs(save_dir, exist_ok=True)


filename = (f"{systemname}.t00z."f"{yearstr}{monstr}{daystr}.fields.n006.nc")


save_path = os.path.join(save_dir, filename)

# download nowcast hour 006 (n006)
bucket = "noaa-nos-ofs-pds"

key = f"{systemname}/netcdf/{yearstr}/{monstr}/{daystr}/{filename}"
#print(key)

# Download file directly to Google Drive
s3.download_file(bucket, key, save_path)


# **Part 2	Understand the FVCOM unstructured mesh**
## **2.1 Read the data file**
Open and read the downloaded data file using xarray, and take a quick look at the data contents by printing the dimension and variable long names. There are a number of variables.



In [ ]:
# open and read data by xarray
ds=xr.open_dataset(save_path)

# print initial time and dimensions
print("------------------------------------")
print("initial time:", ds.time.values[0])
print("zeta:", ds.zeta.shape, ds.zeta.dims)       # (time,node)
print("u:", ds.u.shape, ds.u.dims)                # (time,siglay,nele)
print("temp:", ds.temp.shape, ds.temp.dims)       # (time,siglay,node)
print("------------------------------------")
print(" ")
# print long names of all variables
for var in ds.variables:
    dims = ", ".join(ds[var].dims)
    long_name = ds[var].attrs.get("long_name", "No long_name attribute")
    print(f"{var:<20} dims: {dims:<30} long_name: {long_name}")

##**2.2 Understanding triangular mesh, nodes, elements, and sigma layers**


![link text](https://www.engineering.com/wp-content/uploads/2024/10/img_whatismeshing1.png)

*(Souce: engineering.com)*



Unlike data on a regular rectangular grid, FVCOM model output uses an unstructured triangular mesh. This means the model domain is divided into many connected triangles. Each triangle has nodes, which are the corner points of the triangles, and elements, which represent the triangle cells themselves, often treated as cell centers. In this LEOFS example, the grid contains 6,106 nodes and 11,509 elements.


```text
A single triangular element:

             node
              o
             / \
            /   \
           /     \
     node o-------o node

       triangle cell = element
```

Different variables are stored at different locations on the mesh. Many scalar variables, such as temperature (temp) and water surface elevation (zeta), are defined at the nodes. In contrast, many vector variables, such as the eastward and northward velocity components (u and v), are defined at the elements. This placement arrangement allows an easier implementation of the dinite volume dicrete method over a control volume in numerical computation (see [the FVCOM manual](https://github.com/FVCOM-GitHub/fvcom/blob/main/README.md) and other resources).

``` text
Node-based variable:
temp, zeta

         temp
          o
         / \
        /   \
  temp o-----o temp


Element-based variable:
u, v

         o
        / \
       / ↑ \
      / u,v \
     o-------o
```


This example file contains only one time step, so each time-dependent variable has a time dimension of length 1. The dimensions siglay and siglev describe the model’s vertical coordinate system, called a sigma coordinate system, or a terrain-following coordinate system. Instead of using fixed depth levels, sigma coordinates divide the water column into layers based on fractions of the total water depth. The siglay dimension represents the centers of the vertical layers, while siglev represents the boundaries, or edges, between those layers.

```text
Sigma layers:

Sea or lake surface
────────────────  siglev
       o           siglay
────────────────  siglev
       o           siglay
────────────────  siglev
       o           siglay
────────────────  siglev
Sea or lake floor
```

Some variables are effectively two-dimensional because they vary only horizontally and in time, with no vertical variation (e.g., zeta, aice). These variables do not include siglay or siglev as dimensions. Other variables are three-dimensional because they vary with horizontal position, time, and depth (e.g., temp, u, v). These variables include either the siglay or siglev dimension.


#**Part 3	Plot mesh and surface temperature**

## **3.1 Plotting the Mesh**
Let's plot the triangular mesh. We will use matplotlib for this.

In [ ]:
# Import additional libraries

import matplotlib.tri as mtri
import matplotlib.pyplot as plt

# FVCOM connectivity: nv has shape (three, nele) and is usually 1-based
triangles = (ds["nv"].transpose("nele", "three").values - 1).astype(np.int32)  # (nele, 3)

# Build triangulation from node lon/lat
lon_node = ((ds["lon"].values + 180) % 360) - 180
lat_node = ds["lat"].values

lonc = ((ds["lonc"].values + 180) % 360) - 180
latc = ds["latc"].values

#triang = mtri.Triangulation(ds["lon"].values, ds["lat"].values, triangles)
triang = mtri.Triangulation(lon_node, lat_node, triangles)

fig, ax = plt.subplots(figsize=(7, 7))
ax.triplot(triang, color="k", linewidth=0.2)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("FVCOM triangular mesh (node-based)")

plt.show()

Next, we can make the triangular mesh plot easier to interpret by adding a background map. A background map provides geographic context, so we can see where the model grid is located relative to the Lake Erie shoreline and surrounding land areas.


There are several ways to add map features in Python. In this example, we use GeoPandas together with a basemap/background map layer. Another option is to use built-in map features from Cartopy, such as coastlines and land polygons. However, the default Cartopy features can sometimes be too coarse for regional applications like the Great Lakes, where shoreline details are important. Using GeoPandas or a more detailed basemap can give us a cleaner and more informative map for Lake Erie.

In [ ]:
#! pip install cartopy
!pip -q install geopandas contextily shapely pyproj
#import cartopy.crs as ccrs
import matplotlib.ticker as mticker
#import cartopy.feature as cfeature
import geopandas as gpd
import contextily as cx
from shapely.geometry import box
from pyproj import Transformer


# Lake Erie-ish bbox (lon/lat)
minx, miny, maxx, maxy = -83.6, 41.3, -78.7, 43

# Build bbox polygon in EPSG:4326 then project to EPSG:3857 for web tiles
bbox = gpd.GeoDataFrame(geometry=[box(minx, miny, maxx, maxy)], crs="EPSG:4326").to_crs(epsg=3857)

ax = bbox.plot(figsize=(10, 7), alpha=0)  # invisible polygon just to set extent
ax.set_xlim(*bbox.total_bounds[[0, 2]])
ax.set_ylim(*bbox.total_bounds[[1, 3]])

cx.add_basemap(ax, source=cx.providers.Esri.WorldImagery)

ax.set_axis_off()

# triang: a matplotlib.tri.Triangulation built in lon/lat
lon = triang.x
lat = triang.y
triangles = triang.triangles

# Project lon/lat -> Web Mercator
tfm = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
x3857, y3857 = tfm.transform(lon, lat)

tri3857 = mtri.Triangulation(x3857, y3857, triangles)

# Overlay on the same ax you used for contextily
#ax.triplot(tri3857, color="k", linewidth=0.25, alpha=0.8)
ax.triplot(tri3857, color="w", linewidth=0.2, alpha=0.8)


#plt.show()



Zooming over the western Lake Erie region. You are welcome to play with other background maps. There are several options.

`source=cx.providers.Esri.WorldImagery`

`source=cx.providers.Esri.Esri.WorldStreetMap`

`source=cx.providers.CartoDB.Voyager`

etc etc...


You are also welcome to play with the code to zoom over another areas.

The [supplemental_GLSEA.ipynb](https://colab.research.google.com/drive/1y5JhOoamksgJ6cXIXQo1NN7yLkeGqi5Q?usp=sharing) provides an example to plot a regular grid over the same area (based on GLSEA) for comparison with the triangular mesh.



In [ ]:

tfm = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

# Now define the basemap function
def plot_basemap(ax, bbox_lonlat, source=cx.providers.CartoDB.Positron, zoom=None):
    minlon, minlat, maxlon, maxlat = bbox_lonlat
    xmin, ymin = tfm.transform(minlon, minlat)
    xmax, ymax = tfm.transform(maxlon, maxlat)

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    if zoom is None:
        cx.add_basemap(ax, source=source)
    else:
        cx.add_basemap(ax, source=source, zoom=zoom)

    ax.set_axis_off()
    return ax

# Then create a map for the zoomed (or specified) region.
fig, ax = plt.subplots(figsize=(10, 7))
#plot_basemap(ax, (-83.6, 41.3, -82.2, 42.2), source=cx.providers.Esri.WorldImagery)
plot_basemap(ax, (-83.6, 41.3, -82.2, 42.2), source=cx.providers.CartoDB.Voyager)
#plot_basemap(ax, (-83.6, 41.3, -82.2, 42.2), source=cx.providers.OpenStreetMap.Mapnik)


ax.triplot(tri3857, color="k", linewidth=0.2, alpha=0.8, zorder=10)

plt.show()

## **3.2 Plotting the Lake Surface Temperature**

Next, we will plot a map of lake surface temperature. Let's start from a simple quick plot.

In [ ]:
# start from a simple one
fig, ax = plt.subplots(figsize=(10, 7))
nindex=0 # time index to plot
ax.tripcolor(triang,ds['temp'].isel(siglay=0,time=nindex)) # siglay=0 for surface
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("FVCOM lake surface temperature [degC] "+str(ds['time'].isel(time=nindex).values))
plt.show()

Then plot a little nicer one, zoomed over the western Lake Erie.

In [ ]:
# Then create a ma for the zoomed (or specified) region.
fig, ax = plt.subplots(figsize=(10, 7))

# Western Lake Erie bounding box, lon/lat
bbox_ll = (-83.6, 41.3, -82.2, 42.2)
minlon, minlat, maxlon, maxlat = bbox_ll

# The function created in an earlier cell
#plot_basemap(ax, bbox_ll, source=cx.providers.Esri.WorldImagery)
plot_basemap(ax, bbox_ll, source=cx.providers.CartoDB.Voyager)


p=ax.tripcolor(tri3857,ds['temp'].isel(siglay=0,time=nindex))
ax.triplot(tri3857, color="w", linewidth=0.2, alpha=0.8, zorder=10)
ax.set_title("FVCOM lake surface temperature [degC] "+str(ds['time'].isel(time=nindex).values))

# add a color bar
cb = fig.colorbar(p, ax=ax, fraction=0.035, pad=0.02)
cb.set_label("Temperature ($^oC$)")
plt.show()

# **Part 4	Overlay surface velocity vectors**
## **4.1 Overlay velocity vectors**

Now, we will overlay velocity vectors for the lake surface current. Also, we will use the cmocean package (https://matplotlib.org/cmocean/), which provides nice color schemes for oceanographic variables.

In [ ]:
! pip install cmocean
import cmocean as cmo

fig, ax = plt.subplots(figsize=(10, 7))

# Western Lake Erie bounding box, lon/lat
bbox_ll = (-83.6, 41.3, -82.2, 42.2)
minlon, minlat, maxlon, maxlat = bbox_ll

# Background map
plot_basemap(ax,bbox_ll,source=cx.providers.Esri.WorldImagery)

# Plot surface temperature at siglay=0
p = ax.tripcolor(tri3857,ds["temp"].isel(siglay=0, time=nindex),
                 cmap=cmo.cm.thermal,shading="flat",alpha=0.9)

# Plot triangular mesh
ax.triplot(tri3857,color="w", linewidth=0.2,alpha=0.8, zorder=10)

# Add surface velocity vectors
# Surface current components at element centers
u_surf = ds["u"].isel(siglay=0, time=nindex).values
v_surf = ds["v"].isel(siglay=0, time=nindex).values

# Element-center longitude and latitude
lonc = ds["lonc"].values - 360. # convert from 0~360 to -180~+180
latc = ds["latc"].values

# Keep only vectors inside the plotting region
mask = ((lonc >= minlon) & (lonc <= maxlon) &
    (latc >= minlat) & (latc <= maxlat) &
    np.isfinite(u_surf) & np.isfinite(v_surf))


# Thin the vectors so the plot is not too crowded
# Increase skip for fewer arrows; decrease for more arrows.
skip = 5
idx = np.where(mask)[0][::skip]

# Project lon/lat element centers to Web Mercator, EPSG:3857
tfm = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
xc3857, yc3857 = tfm.transform(lonc[idx], latc[idx])

# Plot velocity vectors
q = ax.quiver(xc3857, yc3857,u_surf[idx],v_surf[idx],
    color="cyan", angles="xy",
    scale_units="xy", width=0.002, scale=1/50000,   # larger denominator = longer arrows
    alpha=0.9,
    zorder=20)

# Add a reference vector
ax.quiverkey(
    q,X=0.86,Y=0.08,
    U=0.3,label="0.3 m/s",
    labelpos="E",coordinates="axes",
    color="cyan",
    labelcolor="cyan")

# Title and colorbar
ax.set_title("FVCOM lake surface temperature and surface current [degC] "
    + str(ds["time"].isel(time=nindex).values))

cb = fig.colorbar(p, ax=ax, fraction=0.035, pad=0.02)
cb.set_label("Temperature ($^\circ$C)")

plt.show()

# **Optional:	Vertical transect of temperature**

Until now, we focused on the lake surface data. It's often important to understand what's going on in the sub-surface. A vertical transect is a useful visualization to evaluate the thermal structure, thermocline and temperature variations in horizontal and vertical directions. In an unstructured mesh data, visualizing a vertical transect involves a few steps, including defining a transect, extracting a nearest model nodes (or elements if you are looking at a vector variable), and interpolating data vertically.

In [ ]:
# load additional libraries
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
from pyproj import Geod

# Define transect endpoints.
start_lon, start_lat = -83.2, 42.1
end_lon, end_lat     = -78.9, 42.9

# Number of points sampled along the transect
ntransect = 300

# Plot settings
cmap = "turbo"
nlevels = 30

def interp_with_nearest_fallback(points, values, xi):
    """
    Interpolate values from unstructured nodes to transect points.
    Uses linear interpolation inside the convex hull and nearest-neighbor
    fallback where linear interpolation returns NaN.
    """
    linear = LinearNDInterpolator(points, values)
    nearest = NearestNDInterpolator(points, values)

    out = linear(xi)

    bad = np.isnan(out)
    if np.any(bad):
        out[bad] = nearest(xi[bad])

    return out


# Create lon/lat points along transect
transect_lon = np.linspace(start_lon, end_lon, ntransect)
transect_lat = np.linspace(start_lat, end_lat, ntransect)

# Points for interpolation
node_points = np.column_stack([lon_node, lat_node])
transect_points = np.column_stack([transect_lon, transect_lat])

# Distance along transect
# Use geodesic distance. Convert transect lon back to -180..180 for pyproj.
transect_lon_for_dist = ((transect_lon + 180) % 360) - 180

geod = Geod(ellps="WGS84")

dist_m = np.zeros(ntransect)
for i in range(1, ntransect):
  # calculate distance between given two points in meters
    _, _, d = geod.inv(
        transect_lon_for_dist[i - 1],
        transect_lat[i - 1],
        transect_lon_for_dist[i],
        transect_lat[i],
    )
    dist_m[i] = dist_m[i - 1] + d

dist_km = dist_m / 1000.0


# Extract variables
temp = ds["temp"].isel(time=nindex).values.astype(float)
siglay = ds["siglay"].values.astype(float)
h = ds["h"].values.astype(float)
zeta = ds["zeta"].isel(time=nindex).values.astype(float)

# temp shape should be: nsiglay x node
nsiglay = temp.shape[0]

# Compute vertical coordinate at nodes for every sigma layer
# z_node has shape nsiglay x node
z_node = zeta + siglay * (h + zeta)

# Optional: interpolate bathymetry and free surface along transect
h_transect = interp_with_nearest_fallback(node_points, h, transect_points)
zeta_transect = interp_with_nearest_fallback(node_points, zeta, transect_points)

# Interpolate temp and depth onto transect
temp_transect = np.empty((nsiglay, ntransect))
z_transect = np.empty((nsiglay, ntransect))

for k in range(nsiglay):
    temp_transect[k, :] = interp_with_nearest_fallback(
        node_points,temp[k, :],transect_points)
    z_transect[k, :] = interp_with_nearest_fallback(
        node_points,z_node[k, :],transect_points)


z_surface = zeta_transect[None, :]
z_bottom = -1. * h_transect[None, :]

# filling the top and bottom with the first and last layers
z_plot = np.vstack([
    z_surface,
    z_transect,
    z_bottom
])

temp_plot = np.vstack([
    temp_transect[0:1, :],
    temp_transect,
    temp_transect[-1:, :]
])

print(z_plot.shape,temp_plot.shape)

# Plot vertical transect

fig, ax = plt.subplots(figsize=(12, 5))

# Distance grid matching temp/z arrays
D = np.tile(dist_km, (nsiglay+2, 1)) # adding 2 for the surface and bottom

cf = ax.contourf(
    D,
    z_plot,
    temp_plot,
    levels=nlevels,
    cmap=cmap,
    extend="both"
)

cbar = fig.colorbar(cf, ax=ax, pad=0.02)
cbar.set_label("Water temperature")

# Plot free surface
ax.plot(dist_km, zeta_transect, color="k", linewidth=0.8, linestyle="--", label="Free surface")

ax.set_xlabel("Distance along transect [km]")
ax.set_ylabel("Elevation [m]")
ax.set_title("Vertical transect of water temperature")

ax.grid(True, alpha=0.25)
ax.legend(loc="best")

plt.tight_layout()
plt.show()

##**Further readings and data sources**

* FVCOM Github repository: https://github.com/FVCOM-GitHub/FVCOM

* FVCOM User manual: https://etchellsfleet27.com/wp-content/uploads/2020/06/FVCOM_User_Manual_v3.1.6.pdf


*   SCHISM utility scripts by James Kessler at NOAA Great Lakes Environmental Research Lab: https://github.com/NOAA-GLERL/SCHISM_grid_utils. Can be adaptable for other unstructured mesh model outputs, such as FVCOM.


* NOAA National Ocean Service Operational Forecast Systems: https://tidesandcurrents.noaa.gov/models.html

* NOAA National Data Buoy Center:
  https://www.ndbc.noaa.gov/

* NOAA CoastWatch Great Lakes Regional Node: https://coastwatch.glerl.noaa.gov/

* NOAA GLERL Great Lakes Surface Environmental Analysis:
  https://coastwatch.glerl.noaa.gov/glsea/

* NOAA NOS OFS public data archive on AWS:
https://registry.opendata.aws/noaa-ofs/